In [19]:
# --- Path設定 ---
import sys
from pathlib import Path

# Project root path.
# This path must be updated to the location where you cloned or extracted this project.
project_root = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project")

# Add src/ as a directory for Python to search for source code
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

#print("sys.path に追加された src:", src_path)
#from ifc_ingestor.ingestor import load_ifc_bundle
#print("IMPORT SUCCESS:", load_ifc_bundle)

In [20]:
from ifc_ingestor.ingestor import load_ifc_bundle
# Path to the input IFC file.(ARC and MEP)
# Update this path according to the location of the IFC file in your directory.
ifc_path_MEP = project_root / "data" / "ifc" / "raw" / "51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01.ifc"
ifc_path_SPACE = project_root / "data" / "ifc" / "raw" / "51M-SKA-00-00-A-ARC-0001_IFC 2x3-P01.ifc"
bundle_MEP = load_ifc_bundle(str(ifc_path_MEP))
bundle_SPACE = load_ifc_bundle(str(ifc_path_SPACE))

print("\n=== IFC Loaded for MEP===")
print("schema:", bundle_MEP.schema)
print("model type:", type(bundle_MEP.model))

print("\n=== IFC Loaded for SPACE===")
print("schema:", bundle_SPACE.schema)
print("model type:", type(bundle_SPACE.model))


=== IFC Loaded for MEP===
schema: IFC2X3
model type: <class 'ifcopenshell.file.file'>

=== IFC Loaded for SPACE===
schema: IFC2X3
model type: <class 'ifcopenshell.file.file'>


In [21]:
# --- BDNS タグ抽出 ---
from bdns_extractor.extractor import extract_bdns_tags, extract_spatial_elements
bdns_assets = extract_bdns_tags(bundle_MEP)

print("\n=== BDNS Extraction Result ===")
print("BDNS-tagged assets:", len(bdns_assets.items))

# 最初の数件を表示
for asset in bdns_assets.items[:10]:
    print(
        "GUID:", asset.ifc_guid,
        "| BDNS:", asset.bdns_tag,
        "| Name:", asset.name,
        "| Class:", asset.raw_ifc_class
    )


=== BDNS Extraction Result ===
BDNS-tagged assets: 6511
GUID: 2HwPth0ED3muP_ET7DKBaz | BDNS: outlet - double switched socket outlet | Name: DSSO-00001 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK99g | BDNS: outlet - double switched socket outlet | Name: DSSO-00002 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9CM | BDNS: outlet - double switched socket outlet | Name: DSSO-00003 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9Ws | BDNS: outlet - double switched socket outlet | Name: DSSO-00004 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9_L | BDNS: outlet - double switched socket outlet | Name: DSSO-00005 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7lq | BDNS: outlet - double switched socket outlet | Name: DSSO-00006 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7qD | BDNS: outlet - double switched socket outlet | Name: DSSO-00007 | Class: IfcFlowTerminal
GUID: 3ID7pTUZ96xeg742Mq2AA7 | BDNS: outlet - double switched socket outlet | Name: DSSO-00008 | Class:

In [22]:
# ============ 空間要素抽出と JSON 出力 ============
from bdns_extractor.extractor import extract_spatial_elements

spatial = extract_spatial_elements(bundle_SPACE)
with open("spatial_elements.json", "w", encoding="utf-8") as f:
    json.dump(
        [e.__dict__ for e in spatial.items],
        f,
        ensure_ascii=False,
        indent=2
    )

In [23]:
from class_mapper.mapper import map_bdns_to_brick_equipment

base_ns = "http://example.com/brick#"

In [25]:
from pathlib import Path
from class_mapper.mapper import map_bdns_to_brick_equipment

project_root = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project")
mapping_csv_path = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\resources\bdns_assets_to_brick_map.csv")

base_ns = "http://example.com/brick#"

equip_set = map_bdns_to_brick_equipment(
    tagged_assets=bdns_assets,
    base_ns=base_ns,
    crosswalk_equip=None,
    mapping_csv_path=str(mapping_csv_path),
)

total = len(equip_set.items)
unmapped = sum(1 for e in equip_set.items if e.extra_props.get("unmapped") is True)
mapped = total - unmapped

print("=== Mapping summary ===")
print("total,", total)
print("mapped,", mapped)
print("unmapped,", unmapped)
print("ratio_unmapped,", f"{(unmapped/total*100):.2f}%" if total else "n/a")

print("\n=== DSSO examples ===")
dssos = [e for e in equip_set.items if isinstance(e.bdns_tag, str) and str(e.bdns_tag).upper().startswith("DSSO")]
for e in dssos[:10]:
    print(
        "GUID:", e.ifc_guid,
        "| brick_class:", e.brick_class,
        "| label:", e.label,
        "| mapping_source:", e.extra_props.get("mapping_source"),
        "| unmapped:", e.extra_props.get("unmapped"),
    )
print("DSSO count,", len(dssos))

=== Mapping summary ===
total, 6511
mapped, 6511
unmapped, 0
ratio_unmapped, 0.00%

=== DSSO examples ===
GUID: 2HwPth0ED3muP_ET7DKBaz | brick_class: brick:Electrical_Outlet | label: DSSO-00001 | mapping_source: code | unmapped: None
GUID: 2HwPth0ED3muP_ET7DK99g | brick_class: brick:Electrical_Outlet | label: DSSO-00002 | mapping_source: code | unmapped: None
GUID: 2HwPth0ED3muP_ET7DK9CM | brick_class: brick:Electrical_Outlet | label: DSSO-00003 | mapping_source: code | unmapped: None
GUID: 2HwPth0ED3muP_ET7DK9Ws | brick_class: brick:Electrical_Outlet | label: DSSO-00004 | mapping_source: code | unmapped: None
GUID: 2HwPth0ED3muP_ET7DK9_L | brick_class: brick:Electrical_Outlet | label: DSSO-00005 | mapping_source: code | unmapped: None
GUID: 1zhYdOoOz9rBBDvGWES7lq | brick_class: brick:Electrical_Outlet | label: DSSO-00006 | mapping_source: code | unmapped: None
GUID: 1zhYdOoOz9rBBDvGWES7qD | brick_class: brick:Electrical_Outlet | label: DSSO-00007 | mapping_source: code | unmapped: Non

In [26]:
equip_set

BrickEquipmentSet(items=[BrickEquipment(ifc_guid='2HwPth0ED3muP_ET7DKBaz', brick_class='brick:Electrical_Outlet', label='DSSO-00001', bdns_tag='DSSO', extra_props={'raw_ifc_class': 'IfcFlowTerminal', 'bdns_label': 'outlet - double switched socket outlet', 'mapping_source': 'code'}), BrickEquipment(ifc_guid='2HwPth0ED3muP_ET7DK99g', brick_class='brick:Electrical_Outlet', label='DSSO-00002', bdns_tag='DSSO', extra_props={'raw_ifc_class': 'IfcFlowTerminal', 'bdns_label': 'outlet - double switched socket outlet', 'mapping_source': 'code'}), BrickEquipment(ifc_guid='2HwPth0ED3muP_ET7DK9CM', brick_class='brick:Electrical_Outlet', label='DSSO-00003', bdns_tag='DSSO', extra_props={'raw_ifc_class': 'IfcFlowTerminal', 'bdns_label': 'outlet - double switched socket outlet', 'mapping_source': 'code'}), BrickEquipment(ifc_guid='2HwPth0ED3muP_ET7DK9Ws', brick_class='brick:Electrical_Outlet', label='DSSO-00004', bdns_tag='DSSO', extra_props={'raw_ifc_class': 'IfcFlowTerminal', 'bdns_label': 'outlet -

In [ ]:
# ベース名前空間は user 提供

BASE_NS = "https://id.ExampleBuilding.org/project#"
tr = TopologyReasoner(base_ns=BASE_NS)
model = tr.load_ifc(str(IFC_PATH))
g = tr.extract_and_map(model)